# Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import os
import glob
import graphical_sampling as gs
import pandas as pd
import numpy as np
import itertools
from tqdm import tqdm
from package_sampling.utils import inclusion_probabilities

/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "XPC_SERVICE_NAME" redefined by R and overriding existing variable. Current: "application.com.jetbrains.pycharm.1003326.4172874", R: "0"
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmpfaczh4", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmp3uJTls"
  warnings.warn(


# Loading and Determining Population

In [2]:
DATA_DIR = "populations"
csv_paths = glob.glob(os.path.join(DATA_DIR, "*.csv"))

coords_dict = {}
probs_dict = {}

for fp in csv_paths:
    name = os.path.splitext(os.path.basename(fp))[0]
    data = np.loadtxt(fp, delimiter=",", skiprows=1)
    coords = data[:, :2]
    probs  = data[:, -1]

    coord_name, prob_name, *rest = name.split("_")
    coord_name = 'cluster' if coord_name == 'clust' else coord_name
    prob_name = 'equal' if prob_name == 'eq' else 'unequal'

    coords_dict[coord_name] = coords
    probs_dict[coord_name] = probs_dict.get(coord_name, {})
    probs_dict[coord_name][prob_name] = probs

print(coords_dict.keys())
print(probs_dict.keys())
print(probs_dict['random'].keys())

dict_keys(['swiss', 'RegularPop1000', 'AggregatedPop1027', 'cluster', 'meuse', 'random', 'grid'])
dict_keys(['swiss', 'RegularPop1000', 'AggregatedPop1027', 'cluster', 'meuse', 'random', 'grid'])
dict_keys(['equal', 'unequal'])


In [3]:
N = 100
n = 5
coords = coords_dict['random']
probs = probs_dict['random']['unequal']
modified_probs = inclusion_probabilities(probs, n=n)
pop = gs.Population(coords, modified_probs)

# Building Initial Designs

In [4]:
orders = [
    # "lexico-yx",
    # "lexico-xy",
    # "random",
    # "angle_0",
    # "distance_0",
    "projection",
    # "center",
    "spiral",
    # "max",
    # "snake",
    # "hilbert",
]

In [5]:
initial_designs = []
combines = list(itertools.product(orders, orders))
num_trials = 2
for units_order, zones_order in tqdm(combines, desc="Generating initial designs", total=len(combines), unit="orders"):
    best = None
    best_score = np.inf
    for _ in range(num_trials):
        ks = gs.sampling.KMeansSampler(
            population=pop,
            n=n,
            n_zones=(2, 2),
            zone_builder='sweep',
            units_order=units_order,
            zones_order=zones_order,
            split_size=0.001
        )
        if ks.expected_moran_score() < best_score:
            best = ks
            best_score = ks.expected_moran_score()

    initial_designs.append(gs.NewDesign(best))

    for _ in range(num_trials):
        ks = gs.sampling.KMeansSampler(
            population=pop,
            n=n,
            n_zones=(1, 1),
            zone_builder='sweep',
            units_order=units_order,
            zones_order=zones_order,
            split_size=0.001
        )
        if ks.expected_moran_score() < best_score:
            best = ks
            best_score = ks.expected_moran_score()

    initial_designs.append(gs.NewDesign(best))


Generating initial designs:   0%|          | 0/4 [00:00<?, ?orders/s]/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmp3uJTls", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmp0wEJ37"
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmp3uJTls", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//RtmpLHPttW"
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" re

In [6]:
for design in initial_designs:
    print(design.kmeans.all_samples.shape, design.kmeans.expected_moran_score())

(114, 5) -0.25371075423144557
(114, 5) -0.25371075423144557
(115, 5) -0.2690932662717034
(115, 5) -0.2690932662717034
(115, 5) -0.23094133780997056
(100, 5) -0.2533234414459326
(114, 5) -0.2724446799431903
(114, 5) -0.2724446799431903


# Run Astar

In [7]:
moran_criteria = gs.criteria.MoranCriteria()

In [8]:
astar = gs.search.AStar(
    initial_designs,
    moran_criteria
)

best initial criteria value -0.2724446799431903


In [9]:
astar.run(
    max_iterations = 1000,
    num_new_nodes = 10,
    max_open_set_size = 1000,

    n_clusters_to_change_order_zone = 0,
    n_changes_in_order_of_zones = 0,

    n_clusters_to_change_order_units = 1,
    n_zones_to_change_order_units = 1,
    n_changes_in_order_of_units = 1,

    n_jobs=-1
)


parent node: -0.2724446799431903
child node: -0.27275668279661675

New best criteria value: -0.27275668279661675

child node: -0.2724446799431903
child node: -0.2727053761073357
child node: -0.27220827895785477
child node: -0.2736109435419647

New best criteria value: -0.2736109435419647

child node: -0.2742864819395676

New best criteria value: -0.2742864819395676

child node: -0.2716413589019546
child node: -0.27105524624538546
child node: -0.2729511271644005

parent node: -0.2742864819395676
child node: -0.27420225567598766
child node: -0.2771976712012515

New best criteria value: -0.2771976712012515

child node: -0.2701051891927465
child node: -0.27400321142034784
child node: -0.2758978445498548
child node: -0.27409299624573286
child node: -0.27290840140976214
child node: -0.27546951125205466
child node: -0.27373016800550404

parent node: -0.2771976712012515
child node: -0.2786779690018786

New best criteria value: -0.2786779690018786

child node: -0.2734204298876758
child node: -

/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmp3uJTls", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//RtmprvCPaY"
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmp3uJTls", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmp9NbOW5"
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders

child node: -0.282063223351042

New best criteria value: -0.282063223351042

child node: -0.2811893133356198
child node: -0.2756043202437686
child node: -0.28024233733199494

parent node: -0.282063223351042
child node: -0.2792601265170098
child node: -0.2824587987497108

New best criteria value: -0.2824587987497108

child node: -0.28125786138264575
child node: -0.28207286963612554
child node: -0.27833369624710524
child node: -0.2818925812279573
child node: -0.28130240461685246

parent node: -0.2824587987497108


/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmp3uJTls", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmp0EnweY"
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmp0EnweY", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//RtmpVdCIwa"
  warnings.warn(


child node: -0.28172033574006633
child node: -0.2796710773423505
child node: -0.28201179280955374
child node: -0.28109274859118866
child node: -0.28014752904565515
child node: -0.28246971006213667

New best criteria value: -0.28246971006213667

child node: -0.27866040502142947
child node: -0.2798400944260527

parent node: -0.28246971006213667
child node: -0.28207413466346787
child node: -0.28220552977777447
child node: -0.27729042294509054
child node: -0.28133080783921505
child node: -0.2823914664010795
child node: -0.28365203240339454

New best criteria value: -0.28365203240339454


parent node: -0.28365203240339454
child node: -0.2834813902803099
child node: -0.282804888852159
child node: -0.28142895689799463
child node: -0.28328086538284686
child node: -0.2825762010622786
child node: -0.28346638152848536
child node: -0.28213697582348113
child node: -0.2773086543037557

parent node: -0.2834813902803099
child node: -0.2796829965520286
child node: -0.2833966738591864
child node: -0.282

KeyboardInterrupt: 

In [10]:
astar.best_criteria_value

-0.2924749076327493

In [11]:
astar.best_design.kmeans.score_summary_df()

,expected,std
measure,,
density,-0.147803,0.156833
moran,-0.292475,0.095070
local_balance,0.665182,1.622909
voronoi,0.135596,0.089773


In [12]:
np.mean(np.abs(astar.best_design.kmeans.fips - probs))

np.float64(0.039529474618581376)

In [13]:
astar.best_design.kmeans.all_samples_probs.sum()

np.float64(1.000000000001)

# Run Bees

In [14]:
bee = gs.search.Bees(
    initial_designs,
    gs.criteria.MoranCriteria(),
    colony_size=20,
    limit=50,
)

Evaluating initial designs...
ABC initialized - Best initial criteria value: -0.2724446799431903


In [15]:
bee.run(
    max_iterations = 1000,

    n_clusters_to_change_order_zone = 0,
    n_changes_in_order_of_zones = 0,

    n_clusters_to_change_order_units = 1,
    n_zones_to_change_order_units = 1,
    n_changes_in_order_of_units = 1,

    n_jobs=-1
)

/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmp3uJTls", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmpl931T1"
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//Rtmp3uJTls", R: "/var/folders/tw/x1njhbdj7q19p65g47mw5dqm0000gn/T//RtmpQSq758"
  warnings.warn(
/Users/mehdi/Documents/Projects/graphical-sampling/.venv/lib/python3.10/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders


Starting ABC with 20 food sources
Colony size: 20, Limit: 50

Iteration 1/1000
NEW BEST at iteration 1
Criteria value: -0.27344141
  Best: -0.273441, Avg: -0.257610

Iteration 2/1000
  Best: -0.273441, Avg: -0.257935

Iteration 3/1000
  Best: -0.273441, Avg: -0.258773

Iteration 4/1000
NEW BEST at iteration 4
Criteria value: -0.27418323
  Best: -0.274183, Avg: -0.259161

Iteration 5/1000
  Best: -0.274183, Avg: -0.259349

Iteration 6/1000
  Best: -0.274183, Avg: -0.259631

Iteration 7/1000
  Best: -0.274183, Avg: -0.260060

Iteration 8/1000
NEW BEST at iteration 8
Criteria value: -0.27594478
  Best: -0.275945, Avg: -0.260495

Iteration 9/1000
  Best: -0.275945, Avg: -0.261022

Iteration 10/1000
NEW BEST at iteration 10
Criteria value: -0.27761509
  Best: -0.277615, Avg: -0.261962

Iteration 11/1000
  Best: -0.277615, Avg: -0.262255

Iteration 12/1000
  Best: -0.277615, Avg: -0.262466

Iteration 13/1000
NEW BEST at iteration 13
Criteria value: -0.27831319
  Best: -0.278313, Avg: -0.263

KeyboardInterrupt: 